# Multiple Linear Regression: Diabetes dataset

1. Load the data
2. Split into train and test
3. Train a model with all features
4. Keep only the significant features and retrain
5. Measure RMSE on the test set

## Step 1. Load the data

442 diabetes patients (built into scikit-learn). We predict **progression**: how much the disease got worse one year later.

Features: age, sex, bmi, bp (blood pressure), total_chol, ldl, hdl, chol_hdl_ratio, log_triglycerides, blood_sugar.

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

df = load_diabetes(as_frame=True, scaled=False).frame
df = df.rename(columns={
    's1': 'total_chol', 's2': 'ldl', 's3': 'hdl',
    's4': 'chol_hdl_ratio', 's5': 'log_triglycerides', 's6': 'blood_sugar',
    'target': 'progression',
})

TARGET = 'progression'
ALL_FEATURES = [c for c in df.columns if c != TARGET]

df.head()

## Step 2. Split into train and test

- **Train (80%)**: the model learns from this.
- **Test (20%)**: hidden until the end, to check the model on data it has never seen.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print('Train:', len(train_df), '  Test:', len(test_df))

## Step 3. Train a model with all features

What to look at in the summary:
- **coef**: how much progression changes when the feature goes up by 1
- **t**: coef ÷ std err. Far from 0 (beyond ±2) = strong evidence
- **P>|t|** (p-value): below 0.05 = significant (keep); above 0.05 = weak (drop)
- **R-squared**: how much of the variation the model explains (0 to 1)
- **Adj. R-squared**: R-squared with a penalty for extra features

In [ ]:
model_all = sm.OLS(train_df[TARGET], sm.add_constant(train_df[ALL_FEATURES])).fit()
print(model_all.summary())

## Step 4. Keep only the significant features and retrain

Keep features with p-value below 0.05, drop the rest, then train again.

Check: is the new **Adj. R-squared** about the same or higher? Then the dropped features weren't helping.

In [ ]:
pvalues = model_all.pvalues.drop('const')
FEATURES = list(pvalues[pvalues < 0.05].index)

print('Keep:', FEATURES)
print('Drop:', list(pvalues[pvalues >= 0.05].index))

model = sm.OLS(train_df[TARGET], sm.add_constant(train_df[FEATURES])).fit()
print(model.summary())

print(f'Adj. R-squared, all features: {model_all.rsquared_adj:.3f}')
print(f'Adj. R-squared, kept features: {model.rsquared_adj:.3f}')

## Step 5. Measure RMSE on the test set

**RMSE** = the typical size of a prediction mistake, in the same units as progression. Lower is better.

We compare it to a **baseline** that always guesses the average. The model should beat it.

In [ ]:
test_pred = model.predict(sm.add_constant(test_df[FEATURES]))
rmse = np.sqrt(mean_squared_error(test_df[TARGET], test_pred))

baseline_pred = [train_df[TARGET].mean()] * len(test_df)
baseline_rmse = np.sqrt(mean_squared_error(test_df[TARGET], baseline_pred))

print(f'Test RMSE     : {rmse:.1f}')
print(f'Baseline RMSE : {baseline_rmse:.1f}   (always guessing the average)')